<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Supervised Learning - Random Forest - Diamonds
</b></font> </br></p>

---


# 0  | Install & Import
***

In [ ]:
# Install

In [ ]:
# Daten
from pandas import read_csv, DataFrame

# Preprocessing
from sklearn.preprocessing import Ordinalcoder

# Modellvorbereitung
from sklearn.model_selection import train_test_split

# Modell (Regression)
from sklearn.ensemble import RandomForestRegressor

# Evaluation
from sklearn.metrics import r2_score, mean_absolute_error

# Visualisierung
import plotly.express as px
import plotly.subplots as sp

# Modell-Analyse / Diagnose
from yellowbrick.regressor import residuals_plot
from yellowbrick.model_selection import feature_importances

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1  | Understand
***


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>
✅ Passenden Transfer-Case als Orientierung auswählen</br>

<p><font color='black' size="5">
Anwendungsfall
</font></p>

---

Dieser klassische Datensatz enthält die Preise und andere Attribute von fast 54.000 Diamanten.



[DataSet](https://www.openml.org/search?type=data&status=active&id=42225)

[Info](https://www.kaggle.com/datasets/shivam2503/diamonds)


In [ ]:
df = read_csv(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/diamonds.csv",
    usecols=[
        "carat",
        "cut",
        "color",
        "clarity",
        "depth",
        "table",
        "price",
    ],
)

In [ ]:
data = df.copy()
target = data.pop("price")

<p><font color='black' size="5">
EDA (Exploratory Data Analysis) mit Pandas
</font></p>

In [ ]:
data.info()

In [ ]:
data.describe().T

In [ ]:
data.groupby("cut").count()

In [ ]:
data.groupby("color").count()

<p><font color='black' size="5">
EDA (Exploratory Data Analysis) mit Plotly
</font></p>

In [ ]:
title_ = "Depth"
b1 = px.box(data["depth"], title=title_, width=600, height=600)

title_ = "Carat"
b2 = px.box(data["carat"], title=title_, width=600, height=600)

title_ = "Table"
b3 = px.box(data["table"], title=title_, width=600, height=600)

fig = sp.make_subplots(rows=1, cols=3, subplot_titles=("Depth", "Carat", "Table"))

for trace in b1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in b2.data:
    fig.add_trace(trace, row=1, col=2)

for trace in b3.data:
    fig.add_trace(trace, row=1, col=3)

# Layout anpassen
fig.update_layout(width=1000, height=500, title_text="Box-Plots")

# Plot anzeigen
fig.show()

# 2 |  Prepare

---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Duplikate ermitteln/löschen</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>
✅ Preprocessing-Entscheidungen aus dem Transfer-Case auf Übertragbarkeit prüfen</br>

<p><font color='black' size="5">
Datentyp ermitteln
</font></p>

In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<p><font color='black' size="5">
Train-Test-Split
</font></p>


In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.2, random_state=42)

<p><font color='black' size="5">
Kodierung
</font></p>

In [ ]:
cat_seq = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["J", "I", "H", "G", "F", "E", "D"],
    ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],]

coder = Ordinalcoder(categories=cat_seq)
coder.fit(data_train[cat_col])
data_train[cat_col] = coder.transform(data_train[cat_col])
data_test[cat_col] = coder.transform(data_test[cat_col])

# 3 |  Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Modellansatz aus dem Transfer-Case als Orientierung für die eigene Lösung nutzen</br>

<p><font color='black' size="5">🌲 Algorithmus: Random Forest (Zufälliger Wald – Regression)</font></p>

Ein Random Forest besteht aus vielen einzelnen Entscheidungsbäumen. Jeder Baum wird auf einer zufälligen Teilmenge der Trainingsdaten und Merkmale trainiert. Bei der Regression ist die finale Vorhersage der Mittelwert aller Baum-Vorhersagen – das macht das Modell robuster als ein einzelner Baum.

**Analogie:** Statt einen einzelnen Experten zu befragen, werden viele unabhängige Experten befragt – der Durchschnitt ihrer Schätzungen ist zuverlässiger.

**Wichtige Parameter:**

| Parameter | Bedeutung |
|-----------|----------|
| `n_estimators` | Anzahl der Bäume im Wald |
| `max_depth` | Maximale Tiefe jedes Baums |
| `max_features` | Zufällig gewählte Merkmale je Split |
| `random_state` | Reproduzierbarkeit |

**In der Praxis relevant wenn:**
- Hohe Genauigkeit wichtiger ist als vollständige Interpretierbarkeit des Modells
- Der Datensatz gemischte Feature-Typen (nach Kodierung), Ausreißer oder fehlende Muster enthält — Random Forests sind robust
- Feature Importance benötigt wird, ohne ein vollständiges Interpretierbarkeitsmodell aufzubauen

**Nicht geeignet wenn:**
- Das Modell auf sehr schwacher Hardware in Echtzeit betrieben werden muss → viele Bäume = höhere Inferenzzeit
- Maximale Genauigkeit auf tabellarischen Daten gefragt ist → dann XGBoost oder LightGBM

**Typischer Fehler:**
`n_estimators` zu niedrig setzen (z.B. 10) — das Ergebnis ist instabil. Faustregel: mindestens 100 Bäume; ab etwa 300 nimmt der Zugewinn durch weitere Bäume kaum noch zu.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Random Forest – Regression</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
flowchart TD
    DATA[Trainingsdaten] --> T1[Baum 1]
    DATA --> T2[Baum 2]
    DATA --> TN[Baum n]
    T1 --> AVG[Mittelwert der Vorhersagen]
    T2 --> AVG
    TN --> AVG
    AVG --> OUT[Finaler Zahlenwert]
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=500))

 <p><font color='black' size="5">
Modellauswahl & Training
</font></p>

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)

In [ ]:
model.fit(data_train, target_train)

# 4 | Evaluate
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Cross-Validation</br>
✅ Bootstrapping</br>
✅ Regularisierung prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>
✅ Ergebnisse mit dem Transfer-Case vergleichen und Unterschiede erklären</br>


<p><font color='black' size="5">
Prediction
</font></p>


In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

<p><font color='black' size="5">
Bestimmtheitsmass
</font></p>

In [ ]:
r2 = r2_score(target_train, target_train_pred)
print(f"Modell: {model} -- Train --- Bestimmtheitsmass: {r2:5.2f}")

In [ ]:
r2 = r2_score(target_test, target_test_pred)
print(f"Modell: {model} -- Test --- Bestimmtheitsmass: {r2:5.2f}")

<p><font color='black' size="5">
Mean Absolut Error
</font></p>

In [ ]:
mae = mean_absolute_error(target_test, target_test_pred)
print(f"Modell: {model} -- Test -- Mean Absolute Error: {mae:5.2f}")

<p><font color='black' size="5">
Aufbau Analysewürfel
</font></p>

In [ ]:
# Übernahme der Testdaten
cube = data_test.copy()
cube.reset_index(inplace=True)

# Übernahme Target real & predict
cube["real"]    = target_test.values
cube["predict"] = target_test_pred

In [ ]:
# Rücktransformation der kodierten und skalierten Features
cube[all_col] = scaler.inverse_transform(cube[all_col])
cube[cat_col] = coder.inverse_transform(cube[cat_col])

<p><font color='black' size="5">
Visualisierung real vs predict
</font></p>

In [ ]:
# Boxplot
title_ = "Boxplot real vs predict"
px.box(cube[["real", "predict"]], title=title_, width=600, height=600)

In [ ]:
# Histogramm
title_ = "Histogramm Prices real vs predict"
fig = px.histogram(cube, x=["real", "predict"], nbins=10, title=title_)
fig.update_layout(barmode="group", bargap=0.2, width=800, height=600)
fig.show()

<p><font color='black' size="5">
Fehlerhafte Vorhersagen
</font></p>

In [ ]:
cube["abs_Abw%"] = abs((cube["real"] - cube["predict"]) / cube["real"] * 100)
%precision 3
cube.head(10).style.format("{:,.1f}", subset=num_col)

In [ ]:
cube.describe().T

In [ ]:
# Histogramm
title_ = "Histogramm absolute Abweichung"
fig = px.histogram(cube, x=["abs_Abw%"], nbins=20, title=title_)
fig.update_layout(barmode="group", bargap=0.2, width=800, height=600)
fig.show()

In [ ]:
_ = residuals_plot(
    model,
    data_train,
    target_train,
    data_test,
    target_test,
    train_color="b",
    test_color="g",
)


<p><font color='black' size="5">
Feature Importance
</font></p>

In [ ]:
px.bar(
    x=model.feature_importances_, y=data.columns, width=1000, height=600
).update_yaxes(categoryorder="total ascending")

In [ ]:
_ = feature_importances(model, data_train, target_train)

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>
✅ Grenzen der Übertragbarkeit vom Transfer-Case auf den eigenen Anwendungsfall dokumentieren</br>